In [68]:
from langgraph.graph import StateGraph,END
import tavily
from langchain_groq import ChatGroq
from IPython.display import display_markdown

In [69]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [70]:
from tavily import TavilyClient
# Initialize Tavily client  
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b"  # you can change this model
)

In [71]:
from typing import TypedDict

class AgentState(TypedDict):
    question: str
    search_results: dict  
    answer: str

In [72]:
def search_web(state: AgentState):
    """
    Node 1: Intelligent Web Search
 
    Uses Tavily's AI-optimized search to find and process web information.
    Behind the scenes: Tavily searches multiple sources, extracts relevant content,
    and uses AI to synthesize the information into a coherent answer.
    """
    print(f"🔍 Searching: {state['question']}")

    search_results = tavily.search(
        query=state["question"],
        max_results=3,           # Number of sources to aggregate
        include_answer=True      # Get AI-generated answer, not just links!
    )

    return {"search_results": search_results}

In [73]:
def summarize_with_llm(state: AgentState):
    """
    Node 2: Summarize Tavily search references into one clear answer.
    Uses Groq (LLaMA 3) or any LLM to synthesize the final response.
    """
    print("🧠 Summarizing references...")

    # Combine all snippets into one context block
    snippets = "\n\n".join(
        f"{item['title']} — {item.get('content', '')}" 
        for item in state["search_results"].get("results", [])
    )

    # Prepare the summarization prompt
    prompt = f"""You are a helpful AI assistant. Based on the following web search results,write a concise and factual summary answering the question below .

    Question: {state['question']}

    Web References:
    {snippets}
    give the answer in record style format.
    Format your answer in 2-3 paragraphs stating the summary in understandable way, and at the end, list the sources.
    """

    # Run LLM summarization
    llm_response = llm.invoke(prompt)

    # Extract and attach sources
    sources = [
        f"- {r['title']}: {r['url']}"
        for r in state["search_results"].get("results", [])
    ]
    final_answer = f"{llm_response.content}\n\nSources:\n" + "\n".join(sources)
    #print(final_answer)

    return {"summarized_answer": final_answer}
    

In [74]:
def create_agent():
    """
    Build the AI Agent Workflow

    This creates a StateGraph where:
    - Each node is an independent function that can read/update shared state
    - LangGraph handles orchestration, state management, and execution flow
    - Easy to extend: just add more nodes and define their connections
    """
    # Create the workflow graph
    workflow = StateGraph(AgentState)

    # Define our processing nodes
    workflow.add_node("search", search_web)        # Step 1: Gather information  
    workflow.add_node("summarize", summarize_with_llm)   # Step 2: Process and format

    # Define the flow of intelligence
    workflow.set_entry_point("search")      # Start here
    workflow.add_edge("search", "summarize")   # After search, go to answer
    workflow.add_edge("summarize", END)        # After answer, we're done

    return workflow.compile()

In [75]:
agent = create_agent()

In [76]:
agent.invoke({"question":"delhi red fort blast"})

🔍 Searching: delhi red fort blast
🧠 Summarizing references...


{'question': 'delhi red fort blast',
 'search_results': {'query': 'delhi red fort blast',
  'follow_up_questions': None,
  'answer': "On November 10, 2025, a car explosion near Delhi's Red Fort killed at least eight people and injured many more. The exact cause remains under investigation. Authorities have invoked antiterror laws to address the incident.",
  'images': [],
  'results': [{'url': 'https://www.cnn.com/2025/11/10/world/new-delhi-explosion-red-fort-intl',
    'title': 'Delhi car blast: India on edge after deadly explosion near Red Fort',
    'content': '# Eight dead in explosion near Red Fort in India’s New Delhi From Reuters Security personnel cordon off the blast site after an explosion near the Red Fort in the old quarters of Delhi on Monday, November 10. Reuters \xa0— At least eight people were killed in an explosion near the landmark Red Fort in a densely populated district of the Indian capital New Delhi, city police said. The blast occurred in a car near the Red Fort,

In [81]:
output = agent.invoke({"question": "delhi red fort blast"})
print(output)


🔍 Searching: delhi red fort blast
🧠 Summarizing references...
{'question': 'delhi red fort blast', 'search_results': {'query': 'delhi red fort blast', 'follow_up_questions': None, 'answer': "On November 10, 2025, a car explosion near Delhi's Red Fort killed at least eight people and injured many more. The exact cause remains under investigation. Authorities have invoked antiterror laws to address the incident.", 'images': [], 'results': [{'url': 'https://www.cnn.com/2025/11/10/world/new-delhi-explosion-red-fort-intl', 'title': 'Delhi car blast: India on edge after deadly explosion near Red Fort', 'content': '# Eight dead in explosion near Red Fort in India’s New Delhi From Reuters Security personnel cordon off the blast site after an explosion near the Red Fort in the old quarters of Delhi on Monday, November 10. Reuters \xa0— At least eight people were killed in an explosion near the landmark Red Fort in a densely populated district of the Indian capital New Delhi, city police said. T